# Supervisor
- 감독관이 지휘를 하듯이 담당 노드들을 고르고 지휘 -> 오케스트레이션
- 지금까지 모은 것을 보고 다시 노드들과 Finish 중에 선택

In [2]:
import os, operator
from typing import Literal, TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from tavily import TavilyClient

fast = init_chat_model("openai:gpt-5.4-nano")   # 지휘 판단(가벼움, 저비용)
main = init_chat_model("openai:gpt-5.6-luna")   # 최종 답(기본)
_tavily = TavilyClient(os.environ["TAVILY_API_KEY"])

DOCS = [   # 2번 파일과 같은 작은 문서 창고(researcher가 여기서 찾는다)
    "연차 유급휴가는 1년간 80퍼센트 이상 출근한 근로자에게 15일이 주어진다.",
    "과정 수료 기준은 출석률 80퍼센트 이상이다.",
    "취업지원 프로그램 신청은 매 학기 초 2주 안에 해야 한다.",
]

In [7]:
# Superviser 다음을 판단
class Sup(BaseModel):
    next: Literal["researcher", "responder", "FINISH"] = Field(
        description="researcher: 아직 근거가 부족할 경우, responder: 근거가 모인다면, Finish: 최종 답 나왔으면"
    )

In [8]:
class GS(TypedDict):
    question: str                              # 원래 질문
    context: Annotated[list, operator.add]     # 조사한 근거를 쌓는 자리(reducer로 이어 붙임)
    answer: str                                # responder가 채울 최종 답
    visited: Annotated[list, operator.add]     # 일한 담당 기록(예: ["researcher","responder"]) — 무한 지휘 방지에 씀
    next: str                                  # supervisor가 정한 '다음 담당'(조건 분기가 이 값을 읽음)
    log: Annotated[list, operator.add]


def supervisor(s: GS):
    done = s.get("visited", [])   # 지금까지 일한 담당 목록
    # ── 로직 안전장치(첫째 브레이크): 모델에 묻기 전에, 끝나야 할 상황을 코드로 못 박는다 ──
    if s.get("answer"):                       # 이미 최종 답이 나왔으면
        return {"next": "FINISH", "log": ["supervisor -> FINISH (답이 나옴)"]}   # 무조건 종료
    if done.count("researcher") >= 2:         # researcher를 두 번 넘게 부르려 하면
        return {"next": "responder", "log": ["supervisor -> responder (조사 상한)"]}   # 강제로 답 작성으로
    # ── 위 두 경우가 아니면, 지휘 판단만 모델에 맡긴다 ──
    decision = fast.with_structured_output(Sup).invoke(
        "너는 팀 지휘자다. 규칙: '이미 일한 담당'에 researcher 가 없으면 researcher 를 고르고, "
        "researcher 가 이미 있으면 responder 를 골라라.\n"
        f"질문: {s['question']}\n"
        f"이미 일한 담당: {done}")
    return {"next": decision.next, "log": [f"supervisor -> {decision.next}"]}

In [9]:
# 📋 붙여넣기 — 담당(worker) 노드
def researcher(s: GS):
    toks = [t for t in s["question"].split() if len(t) >= 2]   # 2글자 이상 토큰만(오탐 방지, 02와 같은 방식)
    doc_hits = [d for d in DOCS if any(t in d for t in toks)]  # 문서에서 먼저 찾고
    web_hits = []
    if not doc_hits:                     # 문서에 없으면 웹으로 보충
        r = _tavily.search(s["question"], max_results=2)
        web_hits = [f"{x['title']}: {x['url']}" for x in r["results"]]
    found = doc_hits + web_hits
    return {"context": found, "visited": ["researcher"],   # 찾은 근거를 context에, 자기 이름을 visited에 남김
            "log": [f"researcher: 문서 {len(doc_hits)} + 웹 {len(web_hits)}건"]}


def responder(s: GS):
    ctx = "\n".join(s.get("context", [])) or "(근거 없음)"   # 쌓인 근거를 한 덩어리로(비면 표시)
    a = main.invoke(f"근거:\n{ctx}\n\n질문: {s['question']}\n근거에 있는 내용으로 한국어로 짧게 답하라.").content
    return {"answer": a, "visited": ["responder"], "log": ["responder: 최종 답 작성"]}

In [13]:
# 그래프 그리기
g = StateGraph(GS)
g.add_node("supervisor", supervisor)
g.add_node("researcher", researcher)
g.add_node("responder", responder)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda s: s["next"],
                        {"researcher":"researcher", "responder":"responder", "FINISH": END})
g.add_edge("researcher", "supervisor")
g.add_edge("responder", "supervisor")

app = g.compile()
print( app.get_graph().draw_ascii() )

                        +-----------+                         
                        | __start__ |                         
                        +-----------+                         
                               *                              
                               *                              
                               *                              
                        +------------+                        
                        | supervisor |                        
                      ..+------------+...                     
                 .....         *         .....                
              ...              *              ...             
           ...                 *                 ...          
+------------+           +-----------+           +---------+  
| researcher |           | responder |           | __end__ |  
+------------+           +-----------+           +---------+  


In [15]:

def run(question):
    # context·visited·log 를 빈 값으로 시작한다(reducer가 이어 붙일 자리를 놓아 줌).
    out = app.invoke({"question": question, "context": [], "visited": [], "log": []},
                     {"recursion_limit": 12})   # 엔진 상한: 노드 12번 넘게 돌면 강제 중단
    print(f"\nQ: {question}")
    for line in out["log"]:
        print("  ·", line)
    print("  방문 담당:", out["visited"])
    print("  A:", out["answer"][:160]) 

run("연차 유급휴가는 며칠입니까")


Q: 연차 유급휴가는 며칠입니까
  · supervisor -> researcher
  · researcher: 문서 1 + 웹 0건
  · supervisor -> responder
  · responder: 최종 답 작성
  · supervisor -> FINISH (답이 나옴)
  방문 담당: ['researcher', 'responder']
  A: 연차 유급휴가는 15일입니다.


In [16]:
run("2026년 9월 기준 파이썬 최신버전과 특징은?")


Q: 2026년 9월 기준 파이썬 최신버전과 특징은?
  · supervisor -> researcher
  · researcher: 문서 1 + 웹 0건
  · supervisor -> responder
  · responder: 최종 답 작성
  · supervisor -> FINISH (답이 나옴)
  방문 담당: ['researcher', 'responder']
  A: 제시된 근거만으로는 2026년 9월 기준 파이썬 최신 버전과 특징을 알 수 없습니다.
